In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from torch.utils.data import DataLoader, random_split
from src.transforms import test_transforms
from src.dataset import ImageDataset
from src.configs import BATCH_SIZE, SEED
from pathlib import Path
from src.model import Model
import torch

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# load best checkpoint
best_checkpoint_path = Path("../outputs/checkpoints/checkpoint_epoch_70.pth")
best_checkpoint = torch.load(best_checkpoint_path)

# load parameters into the model
model = Model().to(device, non_blocking=True)
model.load_state_dict(best_checkpoint["model_state_dict"])

# load test dataset and create dataloader
annot_file_test = Path("../data/preprocessed/test/annotations.csv")
img_dir_test = Path("../data/preprocessed/test/Images")

test_dataset = ImageDataset(annot_file_test, img_dir_test,
                            transform=test_transforms)

generator_ = torch.Generator().manual_seed(SEED)
test_dataset, _ = random_split(test_dataset, [0.2, 0.8], generator=generator_)

test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4,
                     pin_memory=True, persistent_workers=True)

In [4]:
len(test_dl)

31

In [5]:
from torch import round as rd

def pred_coords_labels(postprocessed_preds, pred_coords_list, pred_labels):
    for class_name, preds in postprocessed_preds.items():
        for pred in preds:
            x1 = int(rd(pred[1]))
            y1 = int(rd(pred[2]))
            x2 = int(rd(pred[3]))
            y2 = int(rd(pred[4]))
            pred_coords_list.append((x1, y1, x2, y2))
            pred_labels.append(class_name)

In [6]:
from src.configs import S, IDX_TO_CLASS
from src.utilities import convert_xywh_coordinates

def truth_coords_labels(truth_items, truth_coords_list, truth_labels):
    for i in range(S):
        for j in range(S):
            class_label = IDX_TO_CLASS[int(truth_items[i][j].argmax())]
            coord = truth_items[i][j][20:24]
            
            if torch.any(coord):
                truth_coords_list.append(
                    convert_xywh_coordinates(coord, i, j, True)
                )
                truth_labels.append(class_label)

In [7]:
from src.visualization import draw_rectangles
from matplotlib import pyplot as plt
import matplotlib
matplotlib.use('Agg')
import random

def draw_and_save_image(image, coords_list, labels, name, num):
    fig, ax = plt.subplots(1, figsize=(8, 8))

    image_draw = draw_rectangles(image, coords_list, labels)
    ax.imshow(image_draw)
    ax.axis(False)

    image_dir = Path("../outputs/bounded_images")
    image_dir.mkdir(parents=True, exist_ok=True)

    file_path = image_dir / (str(num) + name)
    
    plt.savefig(file_path, bbox_inches='tight', pad_inches=0)
    plt.close()

In [8]:
from src.transforms import revert_normalization, revert_standardization
from src.postprocessing import postprocess_preds

model.eval()
with (torch.no_grad()):
    for X_batch, y_batch in test_dl:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        # 1. forward pass
        preds_batch = model(X_batch)

        # 2. process preds, prepare bounding boxes, and save images
        for image, truth_item, preds in zip(X_batch, y_batch, preds_batch):
            pred_coords_list, pred_labels = [], []
            truth_coords_list, truth_labels = [], []

            revert_standardization(image)
            image = revert_normalization(image).cpu()
            
            postprocessed_preds = postprocess_preds(preds)

            pred_coords_labels(postprocessed_preds, pred_coords_list, pred_labels)
            truth_coords_labels(truth_item, truth_coords_list, truth_labels)

            num = random.randint(1, 100000)
            
            if pred_coords_list and truth_coords_list:
                draw_and_save_image(image.clone().detach(), pred_coords_list, pred_labels, "pred", num)
                draw_and_save_image(image.clone().detach(), truth_coords_list, truth_labels, "truth", num)

KeyboardInterrupt: 